# שאילתה מהירה — ביצועי מפעיל

נוטבוק אינטראקטיבי: בוחרים מפעיל וטווח תאריכים ומקבלים דוח ביצועים.

כל המפעילים נשלפים **דינמית** מה-API — שום `operator_ref` אינו קבוע בקוד.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # repo root on path
import pandas as pd
from stride_analysis.data.stride_client import StrideClient
from stride_analysis.analysis import calc_execution, calc_penalties, load_penalty_tables
from stride_analysis.reports.generator import generate_report

client = StrideClient()

## 1. גילוי מפעילים — מי קיים?

In [ ]:
operators = client.list_operators(date='2026-05-28')
pd.DataFrame([o.__dict__ for o in operators]).head(20)

## 2. בחירת פרמטרים
שנו את `OPERATOR_REF` לכל ref מהטבלה למעלה (למשל 7=דן? לא — בדקו בטבלה). `None` ⇒ כל המפעילים.

In [ ]:
OPERATOR_REF = 5          # דן (לדוגמה) — שנו לפי הטבלה למעלה; None = כל המפעילים
DATE_FROM = '2026-05-24'
DATE_TO   = '2026-05-30'
ROUTE     = None          # מספר קו ציבורי לסינון, או None

## 3. שליפת נסיעות (תכנון מול ביצוע)

In [ ]:
rides = client.fetch_rides(DATE_FROM, DATE_TO, operator_ref=OPERATOR_REF,
                           route_short_name=ROUTE)
print(f'{len(rides):,} ride rows')
rides.head()

## 4. חישוב ביצוע + קנסות

In [ ]:
summary = calc_execution(rides, group_by=['operator_ref','operator_name','service_date'])
summary = calc_penalties(summary, load_penalty_tables())
summary

## 5. דוח עברית (Markdown)

In [ ]:
from IPython.display import Markdown
md = generate_report(summary, date_from=DATE_FROM, date_to=DATE_TO,
                     title='דן' if OPERATOR_REF else 'כלל המפעילים',
                     daily=summary, request_count=len(client.request_log))
Markdown(md)